# 🛰️ Module 2: Understanding Satellite Imagery

Welcome to **Module 2** of the **Deforestation Detection from Satellite Images** project!

### 🎯 Learning Objectives
1. **Understand Raster Data & GeoTIFFs**: Learn how multi-band spatial grids, geotransforms, and Coordinate Reference Systems (CRS) work.
2. **Explore Multispectral Bands**: Inspect the 5 critical optical bands: **Blue (490nm)**, **Green (560nm)**, **Red (665nm)**, **NIR (842nm)**, and **SWIR (1610nm)**.
3. **Learn the Biophysics of Vegetation**: Understand why healthy photosynthetic canopy absorbs Red & Blue while strongly reflecting NIR and absorbing SWIR.
4. **Build Band Composites**: Create **True Color RGB**, **Color Infrared (CIR)**, and **SWIR Moisture** composites to detect deforestation clearings and logging roads.
5. **Analyze Spectral Signatures**: Plot spectral reflectance curves distinguishing forest, deforested bare soil, water, and roads.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import rasterio

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from modules.module_02_satellite_imagery.dataset_generator import create_sample_multispectral_geotiff
from modules.module_02_satellite_imagery.spectral_inspector import SatelliteImageryInspector

%matplotlib inline
print("✅ Libraries imported successfully! Rasterio version:", rasterio.__version__)

## 1. Load Satellite Imagery & Inspect Raster Metadata

A satellite GeoTIFF is not just a standard image. It contains:
- **Coordinate Reference System (CRS)**: Links pixels to geographic coordinate space (e.g. UTM Zone 20N, EPSG:32620).
- **Affine Transform**: Maps raster pixel coordinates `(row, col)` to real-world easting and northing coordinates `(x, y)`.
- **Multiple Spectral Bands**: Each band captures electromagnetic radiation at a distinct wavelength range.

In [ ]:
tif_path = "../data/samples/sentinel2_sample_scene.tif"
if not os.path.exists(tif_path):
    print("Generating sample GeoTIFF...")
    tif_path = create_sample_multispectral_geotiff(tif_path)

# Initialize inspector
inspector = SatelliteImageryInspector(tif_path)
inspector.print_metadata()

## 2. Statistical Profiles of Spectral Bands

Let's inspect the distribution of surface reflectance across all 5 bands.

In [ ]:
inspector.print_band_statistics()

# Plot histograms
inspector.plot_histograms()

## 3. Individual Spectral Band Inspection (Grayscale)

Notice how the appearance of the landscape shifts drastically across spectral bands:
- **Red & Blue**: Forest canopy appears very dark because chlorophyll absorbs photons to drive photosynthesis.
- **Near-Infrared (NIR)**: Forest canopy appears glowing white/bright because healthy spongy mesophyll cells scatter NIR radiation.
- **Short-Wave Infrared (SWIR)**: Dry bare soil and deforested clearings reflect brightly, whereas healthy leaf water absorbs SWIR.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 5), constrained_layout=True)
for ax, (b_name, b_data) in zip(axes, inspector.band_dict.items()):
    im = ax.imshow(b_data, cmap="gray", vmin=0.0, vmax=float(np.percentile(b_data, 99.5)))
    ax.set_title(b_name, fontsize=12, fontweight="bold", pad=8)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()

## 4. Band Composites (True Color vs. False Color CIR vs. SWIR Agriculture)

Because human eyes can only perceive Red, Green, and Blue, remote sensing scientists map invisible bands into RGB color channels:
1. **True Color (R: Red, G: Green, B: Blue)**: Natural color representation.
2. **Color Infrared CIR (R: NIR, G: Red, B: Green)**: Forest canopy shines brilliant **crimson red**. Deforestation and bare soil stand out in **cyan / grey**.
3. **SWIR Composite (R: SWIR, G: NIR, B: Red)**: Separates moisture and canopy health. Forest is **vivid green**, while dry cleared soil is **copper / orange**.

In [ ]:
rgb = inspector.create_rgb_composite()
cir = inspector.create_false_color_cir()
swir = inspector.create_swir_composite()

fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

axes[0].imshow(rgb)
axes[0].set_title("True Color Composite (RGB)\n[Human Eye View]", fontsize=12, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(cir)
axes[1].set_title("False Color CIR (NIR, Red, Green)\n[Forest = Bright Red | Deforestation = Cyan]", fontsize=12, fontweight="bold", color="darkred")
axes[1].axis("off")

axes[2].imshow(swir)
axes[2].set_title("SWIR Composite (SWIR, NIR, Red)\n[Forest = Green | Cleared Soil = Orange/Copper]", fontsize=12, fontweight="bold", color="darkorange")
axes[2].axis("off")

plt.show()

## 5. Spectral Reflectance Curves (Biophysical Signatures)

Here we compare the exact spectral curves of:
- **Dense Forest Canopy**
- **Deforested / Cleared Soil**
- **Water Body (River)**
- **Logging Road**

In [ ]:
inspector.plot_spectral_signatures()
plt.show()

## 6. Preview of Module 5: Calculating NDVI

The **Normalized Difference Vegetation Index (NDVI)** exploits the contrast between the chlorophyll absorption in the **Red** band and cellular scattering in the **NIR** band:

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

- **Dense Forest**: High NIR (~0.52), Low Red (~0.035) $\rightarrow$ $\text{NDVI} \approx +0.87$
- **Cleared / Bare Soil**: NIR (~0.31), Red (~0.24) $\rightarrow$ $\text{NDVI} \approx +0.13$
- **Water**: Low NIR & Low Red $\rightarrow$ $\text{NDVI} \le 0$

In [ ]:
nir = inspector.nir.astype(np.float32)
red = inspector.red.astype(np.float32)

# Calculate NDVI with epsilon to avoid division by zero
ndvi = (nir - red) / (nir + red + 1e-8)

plt.figure(figsize=(8, 6))
im = plt.imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
plt.title("NDVI Preview: Vegetation (Green) vs. Cleared Land / Water (Yellow/Red)", fontsize=12, fontweight="bold")
plt.axis("off")
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label("NDVI Value", fontsize=10)
plt.show()

print(f"Forest NDVI Mean: {np.mean(ndvi[40:70, 40:70]):.3f}")
print(f"Cleared Land NDVI Mean: {np.mean(ndvi[120:140, 130:150]):.3f}")
print(f"Water NDVI Mean: {np.mean(ndvi[380:400, 240:260]):.3f}")